# Kandinsky V2A — Quality Generalization Across Scenes

## Experiment question

Does the quality advantage you heard from full **Lite** over **Lite Distill** hold across different sound-event types?

This notebook runs both existing checkpoints on the same three five-second video windows. For each scene, the video, short prompt, duration, seed, and inference setup stay fixed; **only the checkpoint changes**.

- No training or fine-tuning.
- No new data collection.
- Three existing CC0 video sources are downloaded automatically.
- Same seed (42) is used within each paired comparison.
- The two outputs are paired per scene; do not compare scene A's audio with scene B's.

This tests checkpoint quality and speed. It does **not** test direct game-token control; Kandinsky still receives a concise text prompt and the video.

## 1 — Check the runtime and choose model storage

Use an A100 80 GB runtime. Both Lite checkpoints are downloaded once. The notebook prefers Colab local-scratch when it has more free space, then stores the Hugging Face cache there.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, urllib.request, random
from pathlib import Path
from urllib.parse import quote
from datetime import datetime, timezone

print("Python:", sys.version)
if sys.version_info < (3, 13):
    raise RuntimeError("This notebook expects the Python 3.13+ Colab runtime used by the working Kandinsky notebook.")

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. Select an A100 GPU and reconnect.")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True
)
print("GPU:", gpu.stdout.strip())

candidates = [Path("/content/local-scratch"), Path("/content")]
CACHE_ROOT = max(
    (p for p in candidates if p.exists()),
    key=lambda p: shutil.disk_usage(p).free
)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024 ** 3)
print(f"Model cache: {CACHE_ROOT} | free: {free_gib:.1f} GiB")
if free_gib < 55:
    raise RuntimeError("At least 55 GiB free space is recommended for both Lite checkpoints. Free space or select a runtime with a larger local-scratch disk.")

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
WORK = Path("/content/kandinsky_quality_generalization")
INPUT_DIR = WORK / "inputs"
WINDOW_DIR = WORK / "windows"
OUTPUT_DIR = WORK / "outputs"
for directory in (INPUT_DIR, WINDOW_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORK)

## 2 — Install the same Kandinsky environment and download both checkpoints

The full Lite model is the quality-focused arm. Lite Distill is the faster reference arm. Each model is generated in a separate command, so model memory is released between runs. This notebook uses PyTorch SDPA, so it installs the official consumer profile directly and skips the optional SageAttention compilation step; Colab may not have the CUDA compiler (`nvcc`) that SageAttention needs.

In [ ]:
REPO = Path("/content/kandinsky-6")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)],
        check=True
    )

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
if not Path("/usr/local/bin/just").exists():
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"],
        check=True
    )

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
print(setup.stdout[-5000:])
if setup.returncode:
    print(setup.stderr[-7000:])
    raise RuntimeError("Kandinsky dependency setup failed. The underlying uv output is shown above.")
SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Kandinsky source commit:", SOURCE_COMMIT)

subprocess.run(["just", "download", "lite-distill", "lite"], cwd=REPO, check=True)
print("Both checkpoints are ready.")

## 3 — Download the same three approved CC0 scenes used in the multi-scene pilot

The selected scenes cover footsteps, continuous water ambience, and a short door event. Each uses its existing short prompt from that pilot. The original video's audio is discarded; Kandinsky receives only the muted five-second video window and the prompt shown in the manifest.

In [ ]:
SCENES = [
    {
        "scene_id": "city_walkers",
        "label": "City pedestrians walking",
        "filename": "Big City Life.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Big_City_Life.webm",
        "source_license": "CC0 1.0",
        "prompt": "Several pedestrians walking on a city pavement, with soft footsteps and distant urban movement. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "running_stream",
        "label": "Water flowing through a stream",
        "filename": "Water_flowing_strem.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Water_flowing_strem.webm",
        "source_license": "CC0 1.0",
        "prompt": "Steady natural water flows over a shallow stream bed, with gentle continuous rippling. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "door_opening",
        "label": "Door opening",
        "filename": "DoorOpen.gif",
        "source_page": "https://commons.wikimedia.org/wiki/File:DoorOpen.gif",
        "source_license": "CC0 1.0",
        "prompt": "A door slowly opens with a short creaking hinge and a subtle latch movement. No music or speech.",
        "start_seconds": 0,
    },
]
DURATION_SECONDS = 5
SEED = 42
USER_AGENT = "Kandinsky-V2A-Quality-Generalization/1.0"

def run_command(command, cwd=None, print_tail=1800):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-print_tail:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-4500:])
        raise RuntimeError("Command failed. Review the output above.")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("ffmpeg and ffprobe are required; they are normally available in Colab.")

SOURCE_METADATA = []
for scene in SCENES:
    encoded_name = quote(scene["filename"], safe="")
    url = "https://commons.wikimedia.org/wiki/Special:Redirect/file/" + encoded_name
    source_path = INPUT_DIR / scene["filename"]
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=180) as response:
        payload = response.read()
        resolved_url = response.geturl()
    if len(payload) < 10_000:
        raise RuntimeError("Download looks incomplete: " + scene["filename"])
    source_path.write_bytes(payload)

    probe = run_command([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(source_path)
    ])
    source_duration = float(probe.stdout.strip())
    start = float(scene["start_seconds"])
    if start + DURATION_SECONDS > source_duration:
        raise ValueError(f"{scene['filename']} is too short for the selected window.")

    scene_video = WINDOW_DIR / f"{scene['scene_id']}.mp4"
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", str(start), "-i", str(source_path), "-t", str(DURATION_SECONDS),
        "-map", "0:v:0", "-an", "-vf", "fps=24,scale='min(960,iw)':-2",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
        "-pix_fmt", "yuv420p", str(scene_video)
    ])

    scene["source_path"] = source_path
    scene["video_path"] = scene_video
    SOURCE_METADATA.append({
        "scene_id": scene["scene_id"],
        "label": scene["label"],
        "source_page": scene["source_page"],
        "source_license": scene["source_license"],
        "resolved_download_url": resolved_url,
        "source_sha256": sha256_file(source_path),
        "source_duration_seconds": source_duration,
        "window_start_seconds": start,
        "window_sha256": sha256_file(scene_video),
        "prompt": scene["prompt"],
    })
    print(f"Ready: {scene['label']} | source {source_duration:.2f}s | 5s window")
print("Prepared", len(SCENES), "paired test scenes.")

## 4 — Generate Lite Distill and full Lite for each scene

The same seed and video window are used for both checkpoints in each pair. We keep the official inference step count: 10 for Lite Distill and 50 for full Lite. Runs are sequential to limit peak memory.

In [ ]:
MODELS = [
    {"id": "lite_distill", "checkpoint": "lite-distill", "offload": "module"},
    {"id": "lite_full", "checkpoint": "lite", "offload": "none"},
]

def extract_wav(input_mp4, output_wav):
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(input_mp4), "-vn", "-ac", "2", "-ar", "44100",
        "-c:a", "pcm_s16le", str(output_wav)
    ])
    if not output_wav.is_file():
        raise FileNotFoundError(output_wav)
    return output_wav

def generate_one(scene, model):
    config_path = REPO / f"quality_generalization_{model['id']}.yaml"
    config_path.write_text(
        f"""checkpoint: {model['checkpoint']}
cache:
  mode: none
offload:
  strategy: {model['offload']}
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
        encoding="utf-8"
    )
    candidate_id = f"{scene['scene_id']}__{model['id']}"
    output_mp4 = OUTPUT_DIR / f"{candidate_id}.mp4"
    output_wav = OUTPUT_DIR / f"{candidate_id}.wav"
    command = [
        "just", "generate", scene["prompt"],
        "--config", str(config_path),
        "--attention-engine", "sdpa",
        "--video-file", str(scene["video_path"]),
        "--time-length", str(DURATION_SECONDS),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]
    print(f"Generating {scene['scene_id']} with {model['id']}...")
    started = time.time()
    run_command(command, cwd=REPO, print_tail=2600)
    if not output_mp4.is_file():
        raise FileNotFoundError(f"Missing Kandinsky output: {output_mp4}")
    elapsed = round(time.time() - started, 2)
    extract_wav(output_mp4, output_wav)
    return {
        "candidate_id": candidate_id,
        "scene_id": scene["scene_id"],
        "model_id": model["id"],
        "checkpoint": model["checkpoint"],
        "prompt": scene["prompt"],
        "seed": SEED,
        "duration_seconds": DURATION_SECONDS,
        "generation_seconds": elapsed,
        "video_file": output_mp4.name,
        "audio_file": output_wav.name,
        "video_sha256": sha256_file(output_mp4),
        "audio_sha256": sha256_file(output_wav),
    }

CANDIDATES = []
for scene in SCENES:
    for model in MODELS:
        CANDIDATES.append(generate_one(scene, model))
    print(f"Pair complete: {scene['label']}")

print("Completed generations:", len(CANDIDATES), "/", len(SCENES) * len(MODELS))

## 5 — Listen to paired outputs and score them

For each scene, listen to Lite Distill and full Lite in either order. Focus on sound quality, naturalness, artifacts, scene match, and synchronization. Treat synchronization as a guardrail: the main question is whether full Lite improves sound quality enough to justify its longer runtime.

In [ ]:
from IPython.display import Audio, display, Markdown

SCORES = {
    item["candidate_id"]: {
        "audio_quality_1_to_5": None,
        "naturalness_1_to_5": None,
        "artifacts_or_noise_1_to_5": None,  # 1 = very little; 5 = very prominent
        "scene_match_1_to_5": None,
        "sync_1_to_5": None,
        "keep_as_is_edit_or_reject": None,
        "notes": "",
    }
    for item in CANDIDATES
}

for scene in SCENES:
    display(Markdown(f"## {scene['label']}"))
    pair = [x for x in CANDIDATES if x["scene_id"] == scene["scene_id"]]
    for item in pair:
        display(Markdown(
            f"### {item['model_id']} — {item['generation_seconds']:.0f}s generation"
        ))
        display(Audio(filename=str(OUTPUT_DIR / item["audio_file"])))

## 6 — Save the comparison package

The report preserves the source and output hashes, prompts, model IDs, generation time, and your listening scores. This is a three-scene pilot, not a general success-rate claim. If Lite only wins on some scene types, that is useful evidence for a future task-based model router.

In [ ]:
REPORT = {
    "experiment": "kandinsky_v2a_quality_generalization_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_repository": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": SOURCE_COMMIT,
    "gpu": gpu.stdout.strip(),
    "seed": SEED,
    "duration_seconds": DURATION_SECONDS,
    "prompt_expansion": "disabled",
    "sources": SOURCE_METADATA,
    "candidates": CANDIDATES,
    "scores": SCORES,
}
REPORT_PATH = WORK / "quality_generalization_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/kandinsky_v2a_quality_generalization.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, "quality_generalization_report.json")
    for item in CANDIDATES:
        archive.write(OUTPUT_DIR / item["audio_file"], "audio/" + item["audio_file"])
        archive.write(OUTPUT_DIR / item["video_file"], "generated_video/" + item["video_file"])
print("Download package:", ZIP_PATH, "|", round(ZIP_PATH.stat().st_size / 1e6, 1), "MB")
from google.colab import files
files.download(str(ZIP_PATH))